# I. Import lib

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Master/AINC/Project/Unifews")

Mounted at /content/drive


In [2]:
!pip install torch_geometric ogb eigency ptflops dotmap powerlaw  -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 17.9 MB/s eta 0:00:00


In [3]:
import random
import numpy as np
import ptflops

import torch
import torch.nn as nn
import torch.optim as optim

from utils.logger import Logger, ModelLogger
from utils.loader import load_edgelist
import utils.metric as metric
from archs import identity_n_norm, flops_modules_dict
import archs.models as models

np.set_printoptions(
    linewidth=160,
    edgeitems=5,
    threshold=20,
    formatter=dict(float=lambda x: "% 9.3e" % x)
)

torch.set_printoptions(
    linewidth=160,
    edgeitems=5
)


# II. Manual configuration

Chỉnh trực tiếp các biến trong cell dưới đây.


In [4]:
# =========================
# Manual configuration
# =========================

seed = 42
dev = 0                  # GPU id, dùng -1 nếu chạy CPU
config = "cora"
algo = "gcn_thr"            # ví dụ: "gcn2", "mlp", hoặc tên layer khác
suffix = ""
thr_a = 0.5
thr_w = 0.5

layer = 2

# Dataset / run config
data = config
path = "./data"
inductive = False
multil = False

# Training config
patience = 100
hidden = 64
dropout = 0.5
lr = 1e-2
weight_decay = 5e-4
epochs = 50


In [5]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if dev >= 0:
    with torch.cuda.device(dev):
        torch.cuda.manual_seed(seed)


In [6]:
if '_' not in algo:
    thr_a, thr_w = 0.0, 0.0

flag_run = f"{seed}-{thr_a:.1e}-{thr_w:.1e}"

logger = Logger(data, algo, flag_run=flag_run)

model_logger = ModelLogger(
    logger,
    patience=patience,
    cmp='max',
    prefix='model' + suffix,
    storage='state_ram' if data in ['cs', 'physics', 'arxiv'] else 'state_gpu'
)

stopwatch = metric.Stopwatch()


In [7]:
adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=data,
    datapath=path,
    inductive=inductive,
    multil=multil,
    seed=seed
)


n=2485, m=12623, F=1433, C=7 | feat: (2485, 1433), label: (2485,) | 1242/621/622=0.50/0.25/0.25


In [8]:
if algo.split('_')[0] in ['gcn2']:
    model = models.SandwitchThr(
        nlayer=layer,
        nfeat=nfeat,
        nhidden=hidden,
        nclass=nclass,
        thr_a=thr_a,
        thr_w=thr_w,
        dropout=dropout,
        layer=algo
    )
elif algo.split('_')[0] in ['mlp']:
    model = models.MLP(
        nlayer=layer,
        nfeat=nfeat,
        nhidden=hidden,
        nclass=nclass,
        thr_w=thr_w,
        dropout=dropout,
        layer='mlp'
    )
else:
    model = models.GNNThr(
        nlayer=layer,
        nfeat=nfeat,
        nhidden=hidden,
        nclass=nclass,
        thr_a=thr_a,
        thr_w=thr_w,
        dropout=dropout,
        layer=algo
    )

model.reset_parameters()
model.kwargs['diag'] = None
diag = model.kwargs['diag']


In [9]:
adj['train'] = identity_n_norm(
    adj['train'],
    edge_weight=None,
    num_nodes=feat['train'].shape[0],
    rnorm=model.kwargs['rnorm'],
    diag=diag
)


In [10]:
adj_train = adj['train']

if isinstance(adj_train, tuple):
    edge_idx = adj_train[0]
else:
    edge_idx = adj_train

print("edge_index shape:", edge_idx.shape)
print("initial num edges:", edge_idx.shape[1])
print("initial tensor numel:", edge_idx.numel())

edge_index shape: torch.Size([2, 12623])
initial num edges: 12623
initial tensor numel: 25246


In [11]:
if logger.lvl_config > 1:
    print(type(model).__name__, algo, thr_a, thr_w)

if logger.lvl_config > 2:
    print(model)

model_logger.register(model, save_init=False)

if dev >= 0:
    model = model.cuda(dev)


GNNThr gcn_thr 0.5 0.5
GNNThr(
  (dropout): Dropout(p=0.5, inplace=False)
  (act): ReLU()
  (convs): ModuleList(
    (0): GCNConvThr(1433, 64)
    (1): GCNConvThr(64, 7)
  )
  (norms): ModuleList(
    (0): BatchNorm1d(64, eps=1e-05, momentum=0.9, affine=True, track_running_stats=True)
  )
)


In [12]:
optimizer = optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    threshold=1e-4,
    patience=15
)

loss_fn = nn.BCEWithLogitsLoss() if multil else nn.CrossEntropyLoss()


In [13]:
def train(x, edge_idx, y, idx_split, epoch, verbose=False):
    model.train()

    if epoch < epochs // 2:
        model.set_scheme('pruneall', 'pruneall')
    else:
        model.set_scheme('pruneall', 'pruneinc')

    x, y = x.cuda(dev), y.cuda(dev)

    if isinstance(edge_idx, tuple):
        edge_idx = (edge_idx[0].cuda(dev), edge_idx[1].cuda(dev))
    else:
        edge_idx = edge_idx.cuda(dev)

    stopwatch.reset()

    stopwatch.start()
    optimizer.zero_grad()
    output = model(x, edge_idx, node_lock=torch.Tensor([]), verbose=verbose)[idx_split]
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()
    stopwatch.pause()

    return loss.item(), stopwatch.time


In [14]:
def eval_model(x, edge_idx, y, idx_split, verbose=False):
    model.eval()
    model.set_scheme('keep', 'keep')

    x, y = x.cuda(dev), y.cuda(dev)

    if isinstance(edge_idx, tuple):
        edge_idx = (edge_idx[0].cuda(dev), edge_idx[1].cuda(dev))
    else:
        edge_idx = edge_idx.cuda(dev)

    calc = metric.F1Calculator(nclass)
    stopwatch.reset()

    with torch.no_grad():
        stopwatch.start()
        output = model(x, edge_idx, node_lock=idx_split, verbose=verbose)[idx_split]
        stopwatch.pause()

        output = output.cpu().detach()
        ylabel = y.cpu().detach()

        if multil:
            output = torch.where(
                output > 0,
                torch.tensor(1, device=output.device),
                torch.tensor(0, device=output.device)
            )
        else:
            output = output.argmax(dim=1)

        calc.update(ylabel, output)

        output = output.numpy()
        ylabel = ylabel.numpy()

    res = calc.compute('macro' if multil else 'micro')
    return res, stopwatch.time, output, y


In [15]:
def cal_flops(x, edge_idx, idx_split, verbose=False):
    model.eval()
    model.set_scheme('keep', 'keep')

    x = x.cuda(dev)

    if isinstance(edge_idx, tuple):
        edge_idx = (edge_idx[0].cuda(dev), edge_idx[1].cuda(dev))
    else:
        edge_idx = edge_idx.cuda(dev)

    handle = model.register_forward_hook(models.GNNThr.batch_counter_hook)
    model.__batch_counter_handle__ = handle

    macs, nparam = ptflops.get_model_complexity_info(
        model,
        (1, 1, 1),
        input_constructor=lambda _: {'x': x, 'edge_idx': edge_idx},
        custom_modules_hooks=flops_modules_dict,
        as_strings=False,
        print_per_layer_stat=verbose,
        verbose=verbose
    )

    return macs / 1e9


In [16]:
with torch.cuda.device(dev):
    torch.cuda.empty_cache()

time_tol = metric.Accumulator()
macs_tol = metric.Accumulator()

epoch_conv = 0
acc_best = 0


In [17]:
for epoch in range(1, epochs + 1):
    verbose = (epoch % 1 == 0) and (logger.lvl_log > 0)

    loss_train, time_epoch = train(
        x=feat['train'],
        edge_idx=adj['train'],
        y=labels['train'],
        idx_split=idx['train'],
        epoch=epoch,
        verbose=verbose
    )
    time_tol.update(time_epoch)

    acc_val, _, _, _ = eval_model(
        x=feat['train'],
        edge_idx=adj['train'],
        y=labels['val'],
        idx_split=idx['val']
    )
    scheduler.step(acc_val)

    macs_epoch = cal_flops(
        x=feat['train'],
        edge_idx=adj['train'],
        idx_split=idx['train']
    )
    macs_tol.update(macs_epoch)

    if verbose:
        res = (
            f"Epoch:{epoch:04d} | "
            f"train loss:{loss_train:.4f}, "
            f"val acc:{acc_val:.4f}, "
            f"time:{time_tol.val:.4f}, "
            f"macs:{macs_tol.val:.4f}"
        )
        if logger.lvl_log > 1:
            logger.print(res)

    acc_best = model_logger.save_best(acc_val, epoch=epoch)

    if model_logger.is_early_stop(epoch=epoch):
        pass
        # break
    else:
        epoch_conv = max(0, epoch - model_logger.patience)


Epoch:0001 | train loss:2.2528, val acc:0.7037, time:1.9705, macs:0.1649
Epoch:0002 | train loss:1.0658, val acc:0.8309, time:1.9784, macs:0.3129
Epoch:0003 | train loss:0.7322, val acc:0.8502, time:1.9852, macs:0.4482
Epoch:0004 | train loss:0.5847, val acc:0.8519, time:1.9919, macs:0.5743
Epoch:0005 | train loss:0.5142, val acc:0.8535, time:1.9988, macs:0.6940
Epoch:0006 | train loss:0.4632, val acc:0.8519, time:2.0058, macs:0.8098
Epoch:0007 | train loss:0.4239, val acc:0.8551, time:2.0131, macs:0.9237
Epoch:0008 | train loss:0.3775, val acc:0.8519, time:2.0199, macs:1.0377
Epoch:0009 | train loss:0.3590, val acc:0.8551, time:2.0268, macs:1.1524
Epoch:0010 | train loss:0.3452, val acc:0.8583, time:2.0378, macs:1.2683
Epoch:0011 | train loss:0.3073, val acc:0.8583, time:2.0493, macs:1.3858
Epoch:0012 | train loss:0.2910, val acc:0.8631, time:2.0605, macs:1.5048
Epoch:0013 | train loss:0.2751, val acc:0.8615, time:2.0731, macs:1.6251
Epoch:0014 | train loss:0.2611, val acc:0.8583, tim

In [18]:
model = model_logger.load('best')

if dev >= 0:
    model = model.cuda(dev)

with torch.cuda.device(dev):
    torch.cuda.empty_cache()


In [19]:
adj['test'] = identity_n_norm(
    adj['test'],
    edge_weight=None,
    num_nodes=feat['test'].shape[0],
    rnorm=model.kwargs['rnorm'],
    diag=model.kwargs['diag']
)


In [20]:
acc_test, time_test, outl, labl = eval_model(
    x=feat['test'],
    edge_idx=adj['test'],
    y=labels['test'],
    idx_split=idx['test']
)

macs_test = cal_flops(
    x=feat['test'],
    edge_idx=adj['test'],
    idx_split=idx['test']
)

numel_a, numel_w = model.get_numel()


In [21]:
print(
    f"[Val] best acc: {acc_best:0.5f} (epoch: {epoch_conv}/{epoch}), "
    f"[Test] best acc: {acc_test:0.5f}",
    flush=True
)

print(
    f"[Train] time: {time_tol.val:0.4f} s "
    f"(avg: {time_tol.avg * 1000:0.1f} ms), "
    f"MACs: {macs_tol.val:0.3f} G (avg: {macs_tol.avg:0.1f} G)"
)
print(
    f"[Test]  time: {time_test:0.4f} s, "
    f"MACs: {macs_test:0.4f} G, "
    f"Num adj: {numel_a:0.3f} k, "
    f"Num weight: {numel_w:0.3f} k"
)


[Val] best acc: 0.86312 (epoch: 0/50), [Test] best acc: 0.86977
[Train] time: 2.4318 s (avg: 48.6 ms), MACs: 5.985 G (avg: 0.1 G)
[Test]  time: 0.0071 s, MACs: 0.1191 G, Num adj: 19.291 k, Num weight: 43.014 k


In [22]:
logger_tab = Logger(data, algo, flag_run=flag_run, dir=('./save', data))
logger_tab.file_log = logger_tab.path_join('log_cs.csv')

hstr, cstr = logger_tab.str_csv(
    data=data,
    algo=algo,
    seed=seed,
    thr_a=thr_a,
    thr_w=thr_w,
    acc_test=acc_test,
    conv_epoch=epoch_conv,
    epoch=epoch,
    time_train=time_tol.val,
    macs_train=macs_tol.val,
    time_test=time_test,
    macs_test=macs_test,
    numel_a=numel_a,
    numel_w=numel_w
)

logger_tab.print_header(hstr, cstr)
print(f"[INFO] CSV log has been written to: {logger_tab.file_log}")


      Data|     Model|  Seed|     ThA|     ThW|    Acc|  Cn|  EP|  Ttrain|  Ctrain|   Ttest|   CTest|  NumelA|  NumelW
cora      ,gcn_thr   ,    42,5.00e-01,5.00e-01,0.86977,   0,  50,  2.4318,   5.985,  0.0071,  0.1191,  19.291,  43.014
[INFO] CSV log has been written to: ./save/cora/log_cs.csv
